# Phân loại hoa 7 lớp với HOG + SVM/KNN

Notebook này thực hiện đầy đủ pipeline cho bài toán phân loại hoa gồm 7 lớp: **Bellflower, Daisy, Dandelion, Lotus, Rose, Sunflower, Tulips**.

Các bước chính:
- Đọc dataset ảnh hoa từ thư mục hiện có trong project.
- Trích xuất đặc trưng HOG cho từng ảnh bằng `scikit-image`.
- Chia dữ liệu train/test.
- Huấn luyện và đánh giá hai mô hình: **HOG + SVM** và **HOG + KNN**.
- Vẽ và lưu các biểu đồ PNG vào thư mục `results/` để đưa vào PowerPoint.

## 1. Import thư viện và cấu hình chung

Notebook sử dụng OpenCV để đọc ảnh, `skimage.feature.hog` để trích xuất HOG, scikit-learn để train/evaluate, matplotlib và seaborn để vẽ biểu đồ.

In [ ]:
from pathlib import Path
import warnings

import cv2
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from skimage.feature import hog
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import StratifiedKFold, learning_curve, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", font_scale=1.05)

PROJECT_ROOT = Path.cwd()
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

CLASSES = ["bellflower", "daisy", "dandelion", "lotus", "rose", "sunflower", "tulip"]
CLASS_DISPLAY = ["Bellflower", "Daisy", "Dandelion", "Lotus", "Rose", "Sunflower", "Tulips"]
IMG_SIZE = (128, 128)
RANDOM_STATE = 42

print("Project root:", PROJECT_ROOT)
print("Results dir:", RESULTS_DIR)

## 2. Tìm và đọc dataset trong project

Code bên dưới tự kiểm tra các thư mục dataset thường gặp. Với cấu trúc repo hiện tại, dataset khả dụng là `preprocessed_outputs/`, trong đó có đủ 7 thư mục lớp. Nếu sau này thêm dataset gốc vào `flower-training/`, notebook sẽ tự ưu tiên thư mục đó.

In [ ]:
def find_dataset_dir(project_root: Path, classes: list[str]) -> Path:
    candidates = [
        project_root / "flower-training",
        project_root / "it3160" / "flower-training",
        project_root / "dataset",
        project_root / "data",
        project_root / "flowers",
        project_root / "preprocessed_outputs",
    ]
    for candidate in candidates:
        if candidate.exists() and all((candidate / cls).is_dir() for cls in classes):
            return candidate
    raise FileNotFoundError(
        "Không tìm thấy dataset có đủ 7 thư mục lớp. "
        "Cần một thư mục chứa: " + ", ".join(classes)
    )


def collect_image_paths(dataset_dir: Path, classes: list[str]) -> tuple[list[Path], list[int]]:
    image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    image_paths = []
    labels = []

    for label, cls in enumerate(classes):
        cls_dir = dataset_dir / cls
        cls_images = sorted([p for p in cls_dir.rglob("*") if p.suffix.lower() in image_exts])
        print(f"{cls:10s}: {len(cls_images)} ảnh")
        image_paths.extend(cls_images)
        labels.extend([label] * len(cls_images))

    if not image_paths:
        raise ValueError(f"Không tìm thấy ảnh trong dataset: {dataset_dir}")

    return image_paths, labels


DATASET_DIR = find_dataset_dir(PROJECT_ROOT, CLASSES)
image_paths, labels = collect_image_paths(DATASET_DIR, CLASSES)

print("\nDataset đang dùng:", DATASET_DIR)
print("Tổng số ảnh:", len(image_paths))

## 3. Trích xuất đặc trưng HOG

Mỗi ảnh được đọc bằng OpenCV, resize về `128 x 128`, chuyển sang grayscale rồi trích xuất HOG. Đặc trưng HOG sau đó được đưa vào SVM và KNN.

In [ ]:
def extract_hog_feature(image_path: Path, img_size: tuple[int, int] = IMG_SIZE) -> np.ndarray:
    image = cv2.imread(str(image_path))
    if image is None:
        raise ValueError(f"Không đọc được ảnh: {image_path}")

    image = cv2.resize(image, img_size, interpolation=cv2.INTER_AREA)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    feature = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        transform_sqrt=True,
        feature_vector=True,
    )
    return feature.astype(np.float32)


X = []
y = []
failed_images = []

for path, label in zip(image_paths, labels):
    try:
        X.append(extract_hog_feature(path))
        y.append(label)
    except Exception as exc:
        failed_images.append((path, str(exc)))

X = np.asarray(X, dtype=np.float32)
y = np.asarray(y, dtype=np.int64)

print("Kích thước ma trận đặc trưng X:", X.shape)
print("Kích thước vector nhãn y:", y.shape)
print("Số ảnh lỗi:", len(failed_images))

if failed_images:
    for path, err in failed_images[:5]:
        print("-", path, "=>", err)

## 4. Chia train/test

Dữ liệu được chia theo tỉ lệ 75% train và 25% test. Tham số `stratify=y` giữ phân phối 7 lớp cân bằng giữa train và test.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Số mẫu train theo lớp:", np.bincount(y_train, minlength=len(CLASSES)))
print("Số mẫu test theo lớp:", np.bincount(y_test, minlength=len(CLASSES)))

## 5. Huấn luyện HOG + SVM và HOG + KNN

Cả hai mô hình đều dùng `StandardScaler` trước classifier vì HOG là vector số thực có nhiều chiều. SVM dùng kernel RBF, KNN dùng `n_neighbors=3` để phù hợp với dataset nhỏ trong repo hiện tại.

In [ ]:
models = {
    "HOG + SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", SVC(kernel="rbf", C=10, gamma="scale", random_state=RANDOM_STATE)),
    ]),
    "HOG + KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", KNeighborsClassifier(n_neighbors=3, weights="distance")),
    ]),
}

predictions = {}

for model_name, model in models.items():
    model.fit(X_train, y_train)
    predictions[model_name] = model.predict(X_test)
    print(f"Đã huấn luyện xong: {model_name}")

## 6. Đánh giá Accuracy, Precision, Recall, F1-score

Bảng dưới đây lấy các metric từ dự đoán thật trên tập test. Ngoài accuracy, precision/recall/F1 được tính theo trung bình macro để mỗi lớp có trọng số như nhau.

In [ ]:
metrics_summary = []

for model_name, y_pred in predictions.items():
    acc = accuracy_score(y_test, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test,
        y_pred,
        average="macro",
        zero_division=0,
    )
    metrics_summary.append({
        "Model": model_name,
        "Accuracy": acc,
        "Precision_macro": precision,
        "Recall_macro": recall,
        "F1_macro": f1,
    })

    print("=" * 70)
    print(model_name)
    print("Accuracy:", f"{acc:.4f}")
    print(classification_report(y_test, y_pred, target_names=CLASS_DISPLAY, zero_division=0))

metrics_summary

## 7. Vẽ và lưu Confusion Matrix

Hai confusion matrix được lưu vào:
- `results/hog_svm_confusion_matrix.png`
- `results/hog_knn_confusion_matrix.png`

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title: str, output_path: Path) -> None:
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(CLASSES)))

    plt.figure(figsize=(9, 7))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=CLASS_DISPLAY,
        yticklabels=CLASS_DISPLAY,
        cbar=False,
    )
    plt.title(title, fontsize=15, weight="bold")
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.xticks(rotation=35, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Đã lưu:", output_path)


plot_confusion_matrix(
    y_test,
    predictions["HOG + SVM"],
    "Confusion Matrix - HOG + SVM",
    RESULTS_DIR / "hog_svm_confusion_matrix.png",
)

plot_confusion_matrix(
    y_test,
    predictions["HOG + KNN"],
    "Confusion Matrix - HOG + KNN",
    RESULTS_DIR / "hog_knn_confusion_matrix.png",
)

## 8. Vẽ và lưu Learning Curve

Learning curve được tính bằng cross-validation có stratify. Đây là kết quả train/evaluate thật trên đặc trưng HOG, không dùng dữ liệu mock.

File xuất ra:
- `results/hog_svm_learning_curve.png`
- `results/hog_knn_learning_curve.png`

In [ ]:
def plot_learning_curve_for_model(model, title: str, output_path: Path) -> None:
    min_class_count = int(np.bincount(y).min())
    n_splits = min(3, min_class_count)
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

    train_sizes, train_scores, valid_scores = learning_curve(
        model,
        X,
        y,
        cv=cv,
        train_sizes=np.linspace(0.35, 1.0, 5),
        scoring="accuracy",
        n_jobs=1,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    train_mean = train_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    valid_mean = valid_scores.mean(axis=1)
    valid_std = valid_scores.std(axis=1)

    plt.figure(figsize=(8.5, 6))
    plt.plot(train_sizes, train_mean, marker="o", linewidth=2, label="Train accuracy")
    plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.18)
    plt.plot(train_sizes, valid_mean, marker="s", linewidth=2, label="Validation accuracy")
    plt.fill_between(train_sizes, valid_mean - valid_std, valid_mean + valid_std, alpha=0.18)
    plt.title(title, fontsize=15, weight="bold")
    plt.xlabel("Số lượng mẫu train")
    plt.ylabel("Accuracy")
    plt.ylim(0, 1.05)
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Đã lưu:", output_path)


plot_learning_curve_for_model(
    models["HOG + SVM"],
    "Learning Curve - HOG + SVM",
    RESULTS_DIR / "hog_svm_learning_curve.png",
)

plot_learning_curve_for_model(
    models["HOG + KNN"],
    "Learning Curve - HOG + KNN",
    RESULTS_DIR / "hog_knn_learning_curve.png",
)

## 9. Biểu đồ so sánh Accuracy giữa SVM và KNN

Biểu đồ này dùng accuracy trên tập test của hai mô hình đã huấn luyện ở trên và lưu vào `results/hog_svm_knn_accuracy_comparison.png`.

In [ ]:
model_names = [row["Model"] for row in metrics_summary]
accuracies = [row["Accuracy"] for row in metrics_summary]

plt.figure(figsize=(7.5, 5.5))
bars = sns.barplot(x=model_names, y=accuracies, palette=["#3f8cff", "#20b486"])
plt.title("So sánh Accuracy: HOG + SVM và HOG + KNN", fontsize=15, weight="bold")
plt.xlabel("Mô hình")
plt.ylabel("Test Accuracy")
plt.ylim(0, 1.05)

for bar, acc in zip(bars.patches, accuracies):
    bars.annotate(
        f"{acc:.3f}",
        (bar.get_x() + bar.get_width() / 2, bar.get_height()),
        ha="center",
        va="bottom",
        fontsize=11,
        weight="bold",
        xytext=(0, 5),
        textcoords="offset points",
    )

comparison_path = RESULTS_DIR / "hog_svm_knn_accuracy_comparison.png"
plt.tight_layout()
plt.savefig(comparison_path, dpi=300, bbox_inches="tight")
plt.show()

print("Đã lưu:", comparison_path)

## 10. Kiểm tra danh sách file kết quả

Cell cuối cùng liệt kê các file PNG đã sinh ra trong `results/`.

In [ ]:
for png_path in sorted(RESULTS_DIR.glob("*.png")):
    print(png_path.name)